# 00. Adquisición y preparación inicial de los datos

Una vez establecida la estructura general del proyecto y definidos los lineamientos de trabajo para garantizar la reproducibilidad del análisis, se inicia la fase de adquisición de datos utilizando el conjunto de datos **MovieLens 32M** como fuente principal de información.

El objetivo de esta etapa consiste en automatizar la preparación inicial de los datos desde el momento de su descarga, reduciendo al mínimo la intervención manual por parte de los integrantes del equipo. De esta manera, se evita depender de operaciones realizadas desde el explorador de archivos del sistema operativo, tales como la creación manual de carpetas, el movimiento de archivos o la reorganización de directorios.

Esta estrategia permite que cualquier miembro del equipo pueda reconstruir la misma estructura de trabajo ejecutando únicamente el código contenido en el proyecto.

### Importación de bibliotecas necesarias

Como primer paso, se importan las bibliotecas que serán utilizadas durante las etapas de adquisición, validación y procesamiento de los datos.

Las dependencias incorporadas cumplen los siguientes propósitos:

- `pathlib.Path`: facilita la gestión de rutas y directorios de forma portable e independiente del sistema operativo.
- `zipfile.ZipFile`: permite manipular archivos comprimidos en formato ZIP, necesarios para la extracción automatizada de los datos descargados.
- `hashlib`: será utilizado posteriormente para verificar la integridad de los archivos mediante funciones hash MD5.
- `pandas`: proporciona estructuras de datos y herramientas para la carga, transformación y análisis tabular.
- `numpy`: incorpora funcionalidades para operaciones numéricas y tratamiento eficiente de grandes volúmenes de datos.

La carga exitosa de estas bibliotecas confirma que el entorno de trabajo se encuentra correctamente configurado y listo para iniciar el proceso de adquisición de datos.

### Configuración de directorios de trabajo

A continuación se definen los directorios que formarán parte del flujo de almacenamiento de información dentro del proyecto. Considerando que estos ya están creados desde la estructuración del proyecto.

Partiendo del directorio raíz del proyecto, se establecen tres ubicaciones principales:

- **data/raw**: almacena los datos originales descargados sin modificaciones.
- **data/processed**: contiene conjuntos de datos transformados o preparados para posteriores análisis.
- **data/external**: reserva destinada a fuentes de información complementarias externas al conjunto principal de datos.

Adicionalmente, se crea un directorio específico denominado:

- **data/raw/ml-32m**

el cual concentrará todos los archivos pertenecientes al conjunto de datos MovieLens 32M.

### Creación automática de la estructura de almacenamiento

Con los directorios definidos, se procede a verificar su existencia y, en caso necesario, crear automáticamente cada una de las carpetas.

Este procedimiento garantiza que:

- La estructura requerida esté disponible antes de iniciar la carga de información.
- El código pueda ejecutarse múltiples veces sin producir errores por directorios existentes.
- Todos los integrantes del equipo trabajen sobre una organización homogénea de datos.
- El proyecto mantenga una separación clara entre datos originales, datos procesados y recursos externos.

### Verificación de las rutas configuradas

Finalmente, se muestran en pantalla las rutas configuradas para el proyecto, permitiendo comprobar visualmente que:

1. La raíz del proyecto ha sido identificada correctamente.
2. El directorio destinado a almacenar los datos originales existe y es accesible.
3. La carpeta específica destinada al conjunto MovieLens 32M ha sido creada correctamente.

Esta verificación constituye un control preliminar importante antes de proceder con las siguientes etapas del flujo de trabajo, tales como la extracción automática de archivos, la validación de integridad de los datos y la carga de los conjuntos de información para su análisis exploratorio.

En conjunto, esta fase establece la infraestructura necesaria para que el proceso de adquisición de datos sea reproducible, automatizado y consistente entre todos los integrantes del equipo.

In [25]:
# 1. IMPORTAMOS LAS BIBLIOTECAS NECESARIAS

from pathlib import Path
from zipfile import ZipFile
import hashlib
import pandas as pd
import numpy as np

print("Bibliotecas importadas correctamente.")

# 2. CONFIGURACIÓN DE DIRECTORIOS

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"

# Directorio específico de MovieLens
MOVIELENS_DIR = RAW_DIR / "ml-32m"

# Crear directorios si no existen
for directory in [
    RAW_DIR,
    PROCESSED_DIR,
    EXTERNAL_DIR,
    MOVIELENS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", PROJECT_ROOT)
print("Datos originales:", RAW_DIR)
print("MovieLens:", MOVIELENS_DIR)


Bibliotecas importadas correctamente.
Raíz del proyecto: c:\Users\jorge\Desktop\DMEMCD_2026\Proyecto_Integrador_M1
Datos originales: c:\Users\jorge\Desktop\DMEMCD_2026\Proyecto_Integrador_M1\data\raw
MovieLens: c:\Users\jorge\Desktop\DMEMCD_2026\Proyecto_Integrador_M1\data\raw\ml-32m


## Localización e inspección preliminar del conjunto de datos

Una vez configurada la estructura de almacenamiento del proyecto, el siguiente paso consiste en localizar y validar el archivo comprimido que contiene el conjunto de datos **MovieLens 32M** descargado previamente.

El propósito de esta etapa es confirmar que el archivo se encuentra disponible en la ubicación esperada y realizar una inspección inicial de su contenido antes de proceder con la extracción automática de los datos.

Este procedimiento constituye un control de calidad preliminar que permite detectar errores de descarga, archivos incompletos o ubicaciones incorrectas antes de avanzar hacia las fases posteriores del análisis.

### Localización del archivo comprimido

En primer lugar, se define la ruta correspondiente al archivo comprimido `ml-32m.zip`, el cual contiene todos los recursos distribuidos dentro del conjunto de datos MovieLens 32M.

Una vez establecida la ruta, se realiza una verificación de existencia para asegurar que el archivo se encuentre disponible en el directorio de descargas previamente definido.

Esta validación persigue varios objetivos:

- Confirmar que la descarga del dataset se realizó correctamente.
- Evitar que procesos posteriores intenten operar sobre archivos inexistentes.
- Detectar rápidamente errores de ubicación o nomenclatura.
- Garantizar que la siguiente fase de extracción cuente con los recursos necesarios.

Como información adicional, se consulta el tamaño del archivo comprimido expresado en megabytes, permitiendo verificar visualmente que la descarga parece consistente con el tamaño esperado del conjunto de datos.

En caso de no localizar el archivo, el proceso se interrumpe mediante una excepción controlada (`FileNotFoundError`), proporcionando un mensaje descriptivo que facilita la identificación del problema.

### Inspección del contenido del archivo ZIP

Tras confirmar la existencia del archivo, se realiza una exploración interna del contenido del paquete comprimido sin necesidad de extraer previamente sus archivos.

Para ello se utiliza la clase `ZipFile`, que permite acceder directamente a la estructura interna del archivo ZIP y obtener un listado de todos los elementos que contiene.

Durante esta inspección se recupera:

- La jerarquía de carpetas incluida en el paquete.
- Los archivos de datos disponibles.
- Los archivos de documentación asociados al dataset.
- Cualquier otro recurso distribuido dentro del conjunto de datos.

La revisión previa del contenido resulta especialmente útil para:

- Comprender la organización interna del paquete.
- Identificar los archivos relevantes para el análisis.
- Confirmar que el dataset descargado corresponde a la versión esperada.
- Planificar el proceso automatizado de extracción y almacenamiento.

### Cuantificación de los elementos contenidos

Finalmente, además del listado detallado de archivos y carpetas, se calcula el número total de elementos contenidos dentro del archivo comprimido.

Esta información proporciona una referencia rápida sobre la complejidad y composición del conjunto de datos, permitiendo corroborar que la estructura interna del paquete coincide con la documentación proporcionada por la fuente de datos.

### Propósito dentro del flujo de adquisición

Esta etapa representa una validación intermedia entre la descarga y la extracción de los datos.

Al completar este proceso se obtiene la certeza de que:

- El archivo comprimido existe y es accesible.
- La descarga se encuentra aparentemente íntegra.
- Se conoce la estructura interna del dataset.
- Se identifican los recursos disponibles para el análisis.
- El conjunto de datos está listo para su extracción automatizada y posterior validación de integridad.

De esta manera se fortalece la reproducibilidad del proyecto y se reduce el riesgo de errores durante las siguientes fases de preparación y análisis de los datos.

In [27]:
# 3. LOCALIZAR ARCHIVO ZIP DE LOS DATOS QUE USAREMOS

# Ruta del archivo ZIP
ZIP_PATH = DOWNLOADS_DIR / "ml-32m.zip"

# Verificar existencia
if ZIP_PATH.exists():
    print("Archivo encontrado:", ZIP_PATH)
    print("Tamaño en MB:", round(ZIP_PATH.stat().st_size / (1024**2), 2))
else:
    raise FileNotFoundError(
        f"No se encontró el archivo: {ZIP_PATH}"
    )

# 4. INSPECCIÓN DEL ARCHIVO ZIP

with ZipFile(ZIP_PATH, "r") as zip_ref:

    file_list = zip_ref.namelist()

    print("Archivos contenidos en el ZIP:\n")

    for file in file_list:
        print(file)

    print("\nNúmero total de elementos:", len(file_list))


Archivo encontrado: C:\Users\jorge\Downloads\ml-32m.zip
Tamaño en MB: 227.88
Archivos contenidos en el ZIP:

ml-32m/
ml-32m/tags.csv
ml-32m/links.csv
ml-32m/README.txt
ml-32m/checksums.txt
ml-32m/ratings.csv
ml-32m/movies.csv

Número total de elementos: 7


## Extracción automatizada y validación de los archivos del conjunto de datos

Una vez inspeccionado el contenido del archivo comprimido y verificada su disponibilidad, se procede a la extracción controlada de los archivos que serán utilizados durante el análisis exploratorio de datos.

El objetivo principal de esta fase consiste en automatizar completamente el proceso de extracción, evitando la manipulación manual de archivos mediante el explorador del sistema operativo. De esta manera, se garantiza que cualquier integrante del equipo pueda reconstruir exactamente el mismo entorno de trabajo ejecutando el Notebook desde el repositorio.

### Definición de los archivos requeridos

Como primer paso, se establece un listado de los archivos esperados dentro del conjunto de datos MovieLens 32M:

- `ratings.csv`
- `tags.csv`
- `movies.csv`
- `links.csv`
- `README.txt`
- `checksums.txt`

Estos archivos constituyen el núcleo del conjunto de datos y contienen tanto la información principal que será analizada como los recursos de documentación y validación necesarios para garantizar la integridad del proceso.

Definir explícitamente los archivos esperados permite establecer un mecanismo de control que detecta automáticamente posibles problemas en la descarga o modificaciones inesperadas en futuras versiones del dataset.

### Búsqueda de archivos dentro de la estructura comprimida

Los archivos distribuidos por MovieLens pueden encontrarse dentro de carpetas internas cuya estructura puede variar entre versiones.

Por esta razón, en lugar de asumir una ruta fija, el procedimiento recorre el contenido completo del archivo ZIP y localiza cada recurso utilizando únicamente su nombre.

Esta estrategia aporta robustez al proceso de extracción ya que:

- Reduce la dependencia de la estructura interna del archivo comprimido.
- Facilita la reutilización del código frente a futuras actualizaciones del dataset.
- Permite identificar automáticamente la ausencia de archivos obligatorios.

En caso de que alguno de los archivos definidos previamente no se encuentre dentro del paquete descargado, el proceso detiene su ejecución mediante una excepción controlada, evitando continuar con un conjunto de datos incompleto.

### Extracción controlada de archivos

Una vez localizados los recursos requeridos, cada archivo es extraído individualmente al directorio destinado a almacenar los datos originales del proyecto.

Durante este proceso se incorpora un mecanismo de protección contra sobrescritura que verifica previamente si el archivo ya existe en el directorio de destino.

Si el recurso ya se encuentra almacenado:

- Se conserva la versión existente.
- Se evita una extracción innecesaria.
- Se reduce el tiempo de ejecución del proceso.

Por el contrario, cuando el archivo aún no existe, éste se copia directamente desde el paquete comprimido hacia la carpeta de trabajo correspondiente.

Esta estrategia permite ejecutar el Notebook múltiples veces sin afectar archivos previamente extraídos ni comprometer la reproducibilidad del proyecto.

### Confirmación del proceso de extracción

A medida que los archivos son procesados, se informa visualmente cuáles han sido extraídos y cuáles ya se encontraban disponibles en el directorio de trabajo.

Esto proporciona una trazabilidad básica del procedimiento y facilita la identificación de posibles incidencias durante la preparación de los datos.

Al finalizar el proceso, se emite una notificación indicando que la extracción se ha completado satisfactoriamente.

### Verificación posterior a la extracción

Una vez copiados los archivos al directorio de MovieLens, se realiza una comprobación del contenido almacenado.

Para cada archivo identificado se obtiene:

- El nombre del recurso.
- Su tamaño en megabytes.
- Su presencia dentro del directorio esperado.

La visualización de esta información permite verificar que:

- Todos los archivos requeridos fueron extraídos correctamente.
- No existen recursos faltantes.
- Los tamaños de los archivos resultan coherentes con el conjunto de datos descargado.
- La estructura de almacenamiento definida para el proyecto se encuentra correctamente organizada.

### Beneficios de esta validación

La ejecución de esta etapa aporta diversas ventajas para el proyecto:

- Automatiza completamente la preparación inicial de los datos.
- Elimina la necesidad de manipulación manual de archivos.
- Garantiza una estructura homogénea entre todos los integrantes del equipo.
- Reduce errores derivados de rutas incorrectas o archivos faltantes.
- Facilita la reproducibilidad del flujo de trabajo.
- Proporciona una primera comprobación de consistencia antes de iniciar las verificaciones de integridad y el análisis exploratorio de los datos.

Como resultado, se dispone de una copia organizada y controlada del conjunto de datos MovieLens 32M, lista para las fases posteriores de validación, carga y exploración analítica.

In [ ]:
# 5. EXTRACCIÓN DE MOVIELENS

EXPECTED_FILES = [
    "ratings.csv",
    "tags.csv",
    "movies.csv",
    "links.csv",
    "README.txt",
    "checksums.txt"
]

with ZipFile(ZIP_PATH, "r") as zip_ref:

    available_files = zip_ref.namelist()

    for filename in EXPECTED_FILES:

        # Localizar el archivo dentro del ZIP,
        # independientemente de su carpeta interna.
        matches = [
            name for name in available_files
            if Path(name).name.lower() == filename.lower()
        ]

        if not matches:
            raise FileNotFoundError(
                f"No se encontró {filename} dentro del ZIP."
            )

        member = matches[0]

        destination = MOVIELENS_DIR / filename

        # Evitar sobrescribir archivos existentes
        if destination.exists():
            print(f"Ya existe: {filename}")
            continue

        # Extraer el contenido directamente al destino
        with zip_ref.open(member) as source:
            with open(destination, "wb") as target:
                target.write(source.read())

        print(f"Extraído: {filename}")

print("\nExtracción finalizada.")

# 6. VERIFICACIÓN DE ARCHIVOS

print("Contenido de MovieLens:\n")

for file in sorted(MOVIELENS_DIR.iterdir()):

    if file.is_file():

        size_mb = file.stat().st_size / (1024**2)

        print(
            f"{file.name:<15} | "
            f"{size_mb:,.2f} MB"
        )

Ya existe: ratings.csv
Ya existe: tags.csv
Ya existe: movies.csv
Ya existe: links.csv
Ya existe: README.txt
Ya existe: checksums.txt

Extracción finalizada.
Contenido de MovieLens:

checksums.txt   | 0.00 MB
links.csv       | 1.86 MB
movies.csv      | 4.05 MB
ratings.csv     | 836.45 MB
README.txt      | 0.01 MB
tags.csv        | 69.00 MB


## Validación de integridad de los archivos mediante hashes MD5

Una vez finalizado el proceso de extracción de los archivos que componen el conjunto de datos MovieLens 32M, se realiza una validación de integridad con el propósito de verificar que los archivos almacenados localmente coinciden con los distribuidos originalmente por los creadores del dataset.

Esta verificación forma parte de las recomendaciones incluidas en la documentación oficial del conjunto de datos y constituye una buena práctica dentro de cualquier proceso de adquisición y preparación de datos.

### Fundamentación de la validación

Durante la descarga, transferencia, copia o extracción de archivos pueden producirse alteraciones involuntarias derivadas de diversos factores, tales como:

- Errores durante la descarga.
- Archivos incompletos.
- Problemas de almacenamiento.
- Corrupción de datos durante la extracción.
- Modificaciones accidentales por parte de algún usuario.

Por este motivo, antes de iniciar cualquier proceso de análisis resulta conveniente comprobar que los archivos obtenidos mantienen exactamente el mismo contenido que los archivos originales publicados por la fuente de datos.

### Uso del archivo checksums.txt

El conjunto de datos MovieLens incluye un archivo denominado `checksums.txt`, que contiene una lista de valores hash asociados a cada uno de los archivos distribuidos dentro del dataset.

Cada registro del archivo almacena:

- El hash MD5 esperado.
- El nombre del archivo correspondiente.

A partir de esta información se construye dinámicamente una estructura de referencia que será utilizada durante la validación.

Este enfoque presenta la ventaja de que evita incorporar manualmente los valores esperados dentro del código y permite que el procedimiento se adapte automáticamente a futuras actualizaciones del conjunto de datos.

### Cálculo de los hashes locales

Una vez identificados los valores de referencia, se calcula el hash MD5 de cada archivo almacenado localmente.

Para ello, los archivos son leídos en bloques de información de tamaño controlado, permitiendo procesar eficientemente incluso archivos de gran tamaño sin necesidad de cargarlos completamente en memoria.

El resultado de este procedimiento es una huella digital única del contenido de cada archivo.

### Comparación entre hashes esperados y obtenidos

Para cada uno de los archivos incluidos en el dataset se comparan dos valores:

- **MD5 esperado**: proporcionado por los creadores del conjunto de datos mediante el archivo `checksums.txt`.
- **MD5 obtenido**: calculado localmente sobre el archivo extraído.

Si ambos valores coinciden, se considera que el archivo conserva íntegramente su contenido original.

En caso contrario, se identifica una discrepancia que podría indicar problemas durante las etapas de descarga, extracción o almacenamiento.

Como resultado de la verificación, se genera una clasificación simple para cada archivo:

- **Correcta**: el hash calculado coincide con el valor esperado.
- **Diferencia**: existe una discrepancia entre ambos valores.

### Generación de un reporte de integridad

Los resultados de la validación se almacenan en una estructura tabular que contiene:

- Nombre del archivo.
- Hash esperado.
- Hash calculado.
- Estado de la validación.

Esta tabla facilita la inspección visual de los resultados y proporciona evidencia documental de que los datos fueron verificados antes de su utilización en las etapas posteriores del proyecto.

### Limitaciones de la validación MD5

Es importante señalar que el algoritmo MD5 no se considera actualmente un mecanismo criptográfico fuerte para aplicaciones relacionadas con seguridad informática debido a la existencia de vulnerabilidades conocidas y posibles colisiones.

Sin embargo, para fines de verificación de integridad en procesos de adquisición de datos, continúa siendo una opción ampliamente utilizada debido a su rapidez de cálculo y a su capacidad para detectar alteraciones accidentales en los archivos.

Por esta razón, aunque la validación mediante MD5 no constituye una comprobación criptográfica robusta, resulta suficiente para los objetivos de este proyecto, especialmente considerando que corresponde al mecanismo de validación recomendado por los propios creadores del conjunto de datos MovieLens.

### Propósito dentro del flujo de preparación de datos

La ejecución de esta etapa permite garantizar que:

- Los archivos descargados coinciden con los publicados por la fuente original.
- No se produjeron alteraciones durante la descarga o extracción.
- Los datos se encuentran en condiciones adecuadas para el análisis.
- El proyecto incorpora controles básicos de calidad de datos desde sus etapas iniciales.

Con esta validación finaliza el proceso de adquisición y preparación inicial del conjunto de datos, dejando los archivos listos para iniciar las actividades de exploración, limpieza y análisis de la información.

In [30]:
# 7. VERIFICACIÓN DE INTEGRIDAD MD5

import hashlib
import pandas as pd

# Leer hashes esperados desde checksums.txt
EXPECTED_MD5 = {}

with open(MOVIELENS_DIR / "checksums.txt", encoding="utf-8") as file:
    for linea in file:
        hash_md5, nombre_archivo = linea.strip().split(maxsplit=1)
        EXPECTED_MD5[nombre_archivo] = hash_md5


def calculate_md5(file_path):
    """Calcula el hash MD5 de un archivo."""

    md5 = hashlib.md5()

    with open(file_path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            md5.update(chunk)

    return md5.hexdigest()


verification_results = []

for filename, expected_hash in EXPECTED_MD5.items():

    file_path = MOVIELENS_DIR / filename

    actual_hash = calculate_md5(file_path)

    is_valid = actual_hash == expected_hash

    verification_results.append({
        "Archivo": filename,
        "MD5 esperado": expected_hash,
        "MD5 obtenido": actual_hash,
        "Integridad": "Correcta" if is_valid else "Diferencia"
    })

    print(
        f"{filename}: "
        f"{'CORRECTO' if is_valid else 'DIFERENCIA'}"
    )

integrity_df = pd.DataFrame(verification_results)

display(integrity_df)

links.csv: CORRECTO
movies.csv: CORRECTO
ratings.csv: CORRECTO
tags.csv: CORRECTO


,Archivo,MD5 esperado,MD5 obtenido,Integridad
0,links.csv,8f033867bcb4e6be8792b21468b4fa6e,8f033867bcb4e6be8792b21468b4fa6e,Correcta
1,movies.csv,0df90835c19151f9d819d0822e190797,0df90835c19151f9d819d0822e190797,Correcta
2,ratings.csv,cf12b74f9ad4b94a011f079e26d4270a,cf12b74f9ad4b94a011f079e26d4270a,Correcta
3,tags.csv,963bf4fa4de6b8901868fddd3eb54567,963bf4fa4de6b8901868fddd3eb54567,Correcta


## Carga, comprensión e inspección inicial de los datos

Una vez verificada la integridad de los archivos extraídos, se procede a la carga de los conjuntos de datos que conforman el dataset **MovieLens 32M**. Esta etapa tiene como objetivo comprender la estructura de la información disponible, identificar las variables contenidas en cada archivo y realizar una primera validación de los tipos de datos antes de iniciar los procesos de limpieza y análisis exploratorio.

La inspección temprana de los datos constituye una práctica fundamental en cualquier proyecto de ciencia de datos, ya que permite identificar posibles inconsistencias, comprender las relaciones entre tablas y conocer el volumen de información que será analizado.

### Configuración de las rutas de acceso

Como primer paso se definen las rutas de los archivos CSV que conforman el dataset:

- `movies.csv`
- `links.csv`
- `tags.csv`
- `ratings.csv`

Centralizar estas rutas facilita la mantenibilidad del código y permite que futuras modificaciones en la ubicación de los archivos puedan realizarse desde un único punto de configuración.

### Carga de la información y optimización de tipos de datos

Durante la lectura de cada archivo se especifican explícitamente los tipos de datos esperados mediante el parámetro `dtype`.

Esta práctica aporta varias ventajas:

- Reduce el consumo de memoria.
- Acelera la lectura de los archivos.
- Evita inferencias incorrectas por parte de Pandas.
- Facilita el control de calidad de las variables.

Cada conjunto de datos cumple una función específica dentro del ecosistema MovieLens:

#### Tabla de películas (`movies`)

Contiene el catálogo de películas disponibles en el sistema de recomendación.

Las principales variables son:

- `movieId`: identificador único de la película.
- `title`: título de la película.
- `genres`: géneros asociados a cada película.

Esta tabla funcionará como catálogo maestro para enriquecer los análisis realizados sobre las calificaciones y etiquetas emitidas por los usuarios.

#### Tabla de identificadores externos (`links`)

Contiene las equivalencias entre los identificadores internos de MovieLens y otros sistemas ampliamente utilizados en la industria cinematográfica.

Entre las variables más relevantes se encuentran:

- `movieId`
- `imdbId`
- `tmdbId`

Estos identificadores permitirían complementar el análisis utilizando fuentes externas de información, como IMDb o The Movie Database (TMDb).

#### Tabla de etiquetas (`tags`)

Almacena las etiquetas o palabras clave asignadas por los usuarios a las películas.

Sus principales variables son:

- `userId`
- `movieId`
- `tag`
- `timestamp`

Estas etiquetas representan información de carácter semántico que puede aportar contexto adicional sobre la percepción de los usuarios respecto a las películas.

#### Tabla de calificaciones (`ratings`)

Contiene las valoraciones numéricas realizadas por los usuarios.

Las variables principales son:

- `userId`
- `movieId`
- `rating`
- `timestamp`

Esta tabla constituye la fuente principal de información para estudiar los patrones de evaluación de los usuarios y la popularidad de las películas.

### Conversión de marcas de tiempo

Un aspecto importante del proceso de carga corresponde al tratamiento de las variables `timestamp` presentes en los archivos `ratings.csv` y `tags.csv`.

Según la documentación oficial del conjunto de datos, estas columnas no almacenan fechas en formato legible, sino que contienen **timestamps Unix**, es decir, valores enteros que representan el número de segundos transcurridos desde el 1 de enero de 1970 a las 00:00:00 UTC (Unix Epoch).

Por ejemplo, un valor como:

```text
1147880044
```

no resulta fácilmente interpretable para un analista. Por esta razón, durante la carga de datos se realiza una transformación utilizando la función `pd.to_datetime()` para convertir estos valores en fechas y horas legibles.

Esta conversión permite:

- Interpretar correctamente cuándo ocurrió una calificación o etiqueta.
- Facilitar futuros análisis temporales.
- Explorar la evolución del comportamiento de los usuarios a lo largo del tiempo.
- Analizar tendencias históricas de participación y consumo.

Además, la conversión se realiza manteniendo la referencia horaria UTC para preservar la coherencia temporal de la información original.

### Inspección inicial de los conjuntos de datos

Una vez cargadas las tablas, se realiza una inspección exploratoria básica de cada dataset.

Para cada conjunto de datos se muestran:

#### Primeros registros

Se visualizan los cinco primeros registros de cada tabla mediante `head()`, permitiendo verificar que la información fue cargada correctamente y comprender el formato general de los datos.

#### Dimensiones

Se consulta el número de filas y columnas presentes en cada archivo mediante la propiedad `shape`.

Esta información permite dimensionar el volumen de datos disponible y anticipar posibles necesidades de procesamiento posteriores.

#### Tipos de datos

Finalmente, se muestran los tipos de datos asociados a cada variable.

La revisión de esta información permite verificar que:

- Los identificadores fueron cargados como variables numéricas.
- Los textos fueron interpretados correctamente.
- Las fechas fueron convertidas exitosamente al formato datetime.
- No existen discrepancias entre los tipos esperados y los obtenidos durante la carga.

### Objetivo de esta etapa

La finalidad de esta fase es garantizar que todos los conjuntos de datos se encuentren correctamente cargados, documentados y comprendidos antes de iniciar los procesos de exploración y análisis.

Al finalizar este procedimiento se dispone de:

- Datos íntegros y correctamente tipificados.
- Marcas temporales convertidas a formatos interpretables.
- Conocimiento preliminar de la estructura de cada tabla.
- Evidencia de que la información se encuentra lista para las etapas de exploración, limpieza, integración y análisis estadístico posteriores.

In [36]:
# 8. RUTAS DE LOS ARCHIVOS

RATINGS_PATH = MOVIELENS_DIR / "ratings.csv"
TAGS_PATH = MOVIELENS_DIR / "tags.csv"
MOVIES_PATH = MOVIELENS_DIR / "movies.csv"
LINKS_PATH = MOVIELENS_DIR / "links.csv"

print("Rutas configuradas correctamente.")

# 9. CARGA DE DATOS

# Películas
movies = pd.read_csv(
    MOVIES_PATH,
    encoding="utf-8",
    dtype={
        "movieId": "int32",
        "title": "string",
        "genres": "string"
    }
)

# Identificadores externos
links = pd.read_csv(
    LINKS_PATH,
    encoding="utf-8",
    dtype={
        "movieId": "int32",
        "imdbId": "Int64",
        "tmdbId": "Int64"
    }
)

# Etiquetas
tags = pd.read_csv(
    TAGS_PATH,
    encoding="utf-8",
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "tag": "string",
        "timestamp": "int64"
    }
)

# Conversión de marca de tiempo Unix a fecha legible
tags["timestamp"] = pd.to_datetime(
    tags["timestamp"],
    unit="s",
    utc=True
)

# Calificaciones
ratings = pd.read_csv(
    RATINGS_PATH,
    encoding="utf-8",
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "rating": "float32",
        "timestamp": "int64"
    }
)

# Conversión de marca de tiempo Unix a fecha legible
ratings["timestamp"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s",
    utc=True
)

# 10. INSPECCIÓN INICIAL DE LOS DATOS

datasets = {
    "PELÍCULAS (movies)": movies,
    "IDENTIFICADORES EXTERNOS (links)": links,
    "ETIQUETAS (tags)": tags,
    "CALIFICACIONES (ratings)": ratings
}

for nombre, tabla in datasets.items():

    print("\n" + "=" * 60)
    print(f"INFORMACIÓN GENERAL: {nombre}")
    print("=" * 60)

    print("\nPrimeros 5 registros:")
    display(tabla.head())

    print(f"Dimensiones: {tabla.shape[0]:,} filas × {tabla.shape[1]} columnas")

    print("\nTipos de datos:")
    display(
        tabla.dtypes
        .rename("Tipo de dato")
        .to_frame()
    )


Rutas configuradas correctamente.

INFORMACIÓN GENERAL: PELÍCULAS (movies)

Primeros 5 registros:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Dimensiones: 87,585 filas × 3 columnas

Tipos de datos:


,Tipo de dato
movieId,int32
title,string[python]
genres,string[python]



INFORMACIÓN GENERAL: IDENTIFICADORES EXTERNOS (links)

Primeros 5 registros:


,movieId,imdbId,tmdbId
0,1,114709,862
1,2,113497,8844
2,3,113228,15602
3,4,114885,31357
4,5,113041,11862


... (6 outputs left out)


INFORMACIÓN GENERAL: CALIFICACIONES (ratings)

Primeros 5 registros:


,userId,movieId,rating,timestamp
0,1,17,4.0,1999-12-03 19:24:37+00:00
1,1,25,1.0,1999-12-03 19:43:48+00:00
2,1,29,2.0,1999-11-22 00:36:16+00:00
3,1,30,5.0,1999-12-03 19:24:37+00:00
4,1,32,5.0,1999-11-22 00:00:58+00:00


Dimensiones: 32,000,204 filas × 4 columnas

Tipos de datos:


,Tipo de dato
userId,int32
movieId,int32
rating,float32
timestamp,"datetime64[ns, UTC]"


## Caracterización inicial y comprensión de la estructura de los datos

Una vez cargados los archivos que conforman el conjunto de datos MovieLens 32M y verificada la correcta interpretación de sus variables, se realiza una caracterización inicial de los datasets con el propósito de comprender su estructura general, evaluar la calidad de los datos disponibles e identificar posibles aspectos que requieran atención durante las etapas posteriores de limpieza y análisis.

Esta fase corresponde a una de las actividades más importantes dentro de la metodología de comprensión de datos, ya que permite obtener una visión global del volumen, consistencia y completitud de la información antes de iniciar análisis más complejos.

### Resumen general de los conjuntos de datos

Como primer paso se genera un resumen consolidado para cada uno de los archivos que componen el dataset:

- `movies`
- `ratings`
- `tags`
- `links`

Para cada tabla se calculan métricas generales que permiten dimensionar la información disponible.

#### Número de registros

Indica la cantidad total de observaciones contenidas en cada dataset.

Esta métrica permite identificar rápidamente cuáles tablas concentran el mayor volumen de información y cuáles funcionan como tablas de referencia o catálogos.

#### Número de columnas

Corresponde a la cantidad de variables disponibles en cada conjunto de datos.

Este indicador proporciona una visión preliminar de la complejidad estructural de cada tabla y del nivel de detalle de información disponible.

#### Valores únicos

Se calcula el número total de valores distintos presentes en las variables de cada dataset.

Esta métrica resulta útil para:

- Evaluar la diversidad de la información.
- Detectar variables con baja variabilidad.
- Identificar posibles variables clave o identificadores.
- Obtener una primera aproximación al nivel de granularidad de los datos.

#### Registros duplicados

Se contabilizan las filas completamente repetidas dentro de cada tabla.

La presencia de observaciones duplicadas puede indicar:

- Problemas durante la captura de información.
- Errores de procesamiento.
- Replicación involuntaria de registros.

La identificación temprana de duplicados permite planificar estrategias de depuración en fases posteriores.

#### Valores nulos

Se calcula el número total de datos faltantes presentes en cada conjunto de datos.

Esta métrica constituye uno de los principales indicadores de calidad de la información y permite identificar posibles limitaciones analíticas derivadas de la ausencia de datos.

#### Porcentaje de completitud

A partir del número de valores faltantes se estima el porcentaje de celdas que contienen información válida respecto al total de observaciones disponibles.

Este indicador resume el nivel general de calidad del dataset y facilita la comparación entre tablas.

#### Consumo de memoria

Finalmente se calcula la memoria utilizada por cada tabla en megabytes.

La evaluación del tamaño de los datos resulta útil para:

- Dimensionar los requerimientos computacionales.
- Optimizar futuras transformaciones.
- Identificar posibles oportunidades de reducción de memoria mediante la selección adecuada de tipos de datos.

### Perfil detallado de las variables

Posteriormente se construye un perfil descriptivo para cada uno de los conjuntos de datos con el objetivo de comprender las características individuales de cada variable.

Para cada columna se reporta:

#### Tipo de dato

Permite verificar que las variables hayan sido interpretadas correctamente durante el proceso de carga.

Esta validación resulta especialmente importante para:

- Identificadores.
- Variables numéricas.
- Variables categóricas.
- Variables temporales.

#### Número de registros no nulos

Indica cuántas observaciones contienen información válida para cada variable.

Esta métrica ayuda a evaluar la disponibilidad efectiva de información para futuros análisis.

#### Número de valores nulos

Permite identificar qué variables presentan ausencia de información y cuantificar la magnitud de esta problemática.

#### Porcentaje de valores nulos

Complementa el recuento absoluto facilitando la comparación entre variables con diferentes distribuciones.

Las variables con altos porcentajes de valores faltantes podrían requerir tratamientos específicos durante las etapas de preparación de datos.

#### Cardinalidad o número de valores únicos

Se calcula el número de categorías o valores distintos presentes en cada variable.

Esta información permite:

- Detectar posibles identificadores únicos.
- Diferenciar variables continuas y categóricas.
- Evaluar el nivel de diversidad de la información.
- Identificar columnas con potencial analítico relevante.

### Análisis de cobertura temporal

Dado que los conjuntos de datos `ratings` y `tags` contienen información temporal, se realiza una exploración específica de sus variables de fecha.

Recordemos que dichas columnas fueron transformadas previamente desde su representación original como timestamps Unix hacia objetos datetime reconocibles por Pandas.

Para cada tabla se analiza:

#### Fecha mínima registrada

Representa el evento más antiguo disponible en el dataset.

Esta información permite identificar el inicio efectivo de la cobertura temporal de los datos.

#### Fecha máxima registrada

Corresponde al evento más reciente registrado.

Su análisis permite conocer hasta qué momento se encuentra actualizada la información disponible.

#### Periodo cubierto

Se calcula la diferencia entre la fecha más reciente y la más antigua.

Esta métrica proporciona una medida de la amplitud temporal del conjunto de datos y resulta particularmente útil para evaluar la viabilidad de futuros análisis longitudinales o estudios de evolución temporal del comportamiento de los usuarios.

### Objetivo de la caracterización inicial

La finalidad de esta etapa es construir una visión integral de la estructura y calidad de los datos antes de iniciar procesos más avanzados de exploración y modelado.

Al finalizar este análisis preliminar se obtiene información sobre:

- El volumen total de información disponible.
- La estructura de cada tabla.
- El nivel de completitud de los datos.
- La existencia de valores duplicados o faltantes.
- La cardinalidad de las variables.
- Los tipos de datos presentes.
- La cobertura temporal de las interacciones registradas.

Estos resultados constituyen la base para planificar de forma informada las siguientes etapas del análisis exploratorio, incluyendo la evaluación de relaciones entre tablas, la detección de anomalías, el estudio de distribuciones y la generación de indicadores descriptivos más avanzados.

In [37]:
# 11. RESUMEN GENERAL DEL DATASET

datasets = {
    "Movies": movies,
    "Ratings": ratings,
    "Tags": tags,
    "Links": links
}

summary = []

for name, df in datasets.items():

    total_cells = df.shape[0] * df.shape[1]
    total_nulls = df.isna().sum().sum()

    summary.append({
        "Dataset": name,
        "Registros": df.shape[0],
        "Columnas": df.shape[1],
        "Valores únicos": df.nunique().sum(),
        "Duplicados": df.duplicated().sum(),
        "Valores nulos": total_nulls,
        "% completitud": round(
            (1 - total_nulls / total_cells) * 100,
            2
        ),
        "Memoria (MB)": round(
            df.memory_usage(deep=True).sum() / (1024**2),
            2
        )
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

,Dataset,Registros,Columnas,Valores únicos,Duplicados,Valores nulos,% completitud,Memoria (MB)
0,Movies,87585,3,176765,0,0,100.00,11.92
1,Ratings,32000204,4,26261913,0,0,100.00,610.36
2,Tags,2000072,4,1499400,0,17,100.00,145.21
3,Links,87585,3,262595,0,124,99.95,1.84


In [ ]:
# Análisis para cada una de los DataSets

for nombre, tabla in datasets.items():

    print("\n" + "=" * 80)
    print(nombre.upper())
    print("=" * 80)

    perfil = pd.DataFrame({
        "Tipo": tabla.dtypes.astype(str),
        "No nulos": tabla.notna().sum(),
        "Nulos": tabla.isna().sum(),
        "% nulos": round(
            tabla.isna().mean() * 100,
            2
        ),
        "Valores únicos": tabla.nunique(dropna=True)
    })

    display(perfil)


MOVIES


,Tipo,No nulos,Nulos,% nulos,Valores únicos
movieId,int32,87585,0,0.0,87585
title,string,87585,0,0.0,87382
genres,string,87585,0,0.0,1798



RATINGS


,Tipo,No nulos,Nulos,% nulos,Valores únicos
userId,int32,32000204,0,0.0,200948
movieId,int32,32000204,0,0.0,84432
rating,float32,32000204,0,0.0,10
timestamp,"datetime64[ns, UTC]",32000204,0,0.0,25976523



TAGS


,Tipo,No nulos,Nulos,% nulos,Valores únicos
userId,int32,2000072,0,0.0,15848
movieId,int32,2000072,0,0.0,51323
tag,string,2000055,17,0.0,140979
timestamp,"datetime64[ns, UTC]",2000072,0,0.0,1291250



LINKS


,Tipo,No nulos,Nulos,% nulos,Valores únicos
movieId,int32,87585,0,0.00,87585
imdbId,Int64,87585,0,0.00,87585
tmdbId,Int64,87461,124,0.14,87425


In [ ]:
# Coberturas temporales que abarca el conjunto de datos

for nombre, tabla in {
    "Ratings": ratings,
    "Tags": tags
}.items():

    print(f"\n--- Cobertura temporal: {nombre} ---")

    print(
        f"Fecha mínima: "
        f"{tabla['timestamp'].min()}"
    )

    print(
        f"Fecha máxima: "
        f"{tabla['timestamp'].max()}"
    )

    print(
        f"Periodo cubierto: "
        f"{(tabla['timestamp'].max() - tabla['timestamp'].min()).days:,} días"
    )


--- Cobertura temporal: Ratings ---
Fecha mínima: 1995-01-09 11:46:44+00:00
Fecha máxima: 2023-10-13 02:29:07+00:00
Periodo cubierto: 10,503 días

--- Cobertura temporal: Tags ---
Fecha mínima: 2005-12-24 13:00:10+00:00
Fecha máxima: 2023-10-12 23:56:23+00:00
Periodo cubierto: 6,501 días
